In [1]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

# functions

from ensemble_chaos_tools import check_chaos
from earthkit.regrid import interpolate

import xarray as xr
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mtickers
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from matplotlib import rcParams
import cartopy.crs as ccrs
import nicopal as ncp
import json

import metpy.calc as mpcalc
from metpy.interpolate import interpolate_to_grid
from metpy.units import units
from pyextremes import EVA
from colindex2 import Detect

from tqdm.notebook import tqdm
import os
import glob
import pickle
import math
import shutil

%load_ext autoreload
%autoreload 2
#%config InlineBackend.figure_format="jpeg"

In [2]:
def detect_cutoffs(
    base_dir, forecast_init_date, target_var, lat_min, lat_max, lon_min, lon_max,
    window_start="2026-05-21", window_end="2026-05-25"
):
    """
    Detect and extract cutoff low pressure systems from forecast data.
    Saves the individual detected systems to a CSV file.
    """
    # load dataset
    pattern = f"{base_dir}/{forecast_init_date}-*/*-{target_var}.nc"
    file_paths = glob.glob(pattern)
    v = xr.open_dataset(file_paths[0], decode_timedelta=True, engine="h5netcdf")
    v = v.assign_coords(longitude=(((v.longitude + 180) % 360) - 180)) / 9.81
    v = v.swap_dims({"step": "valid_time"})

    target_times = pd.date_range(start=window_start, end=window_end, freq="D")
    total_members = len(v.number)

    # extract lat and lon for metpy
    lon_1d = v["longitude"].compute().values
    lat_1d = v["latitude"].compute().values

    final_results = []

    # compute loop, using metpy for regridding and colindex2 for detecting low pressure areas
    for current_ts in target_times:
        day_str = current_ts.strftime("%Y-%m-%d")
        
        v_step = v.sel(valid_time=day_str).mean(dim="valid_time")
        
        ym_str = current_ts.strftime("%Y%m")
        filename = f"V-L-{current_ts.strftime('%Y%m%d0000')}-0500.csv"

        print(f"\n========== Processing Target Date: {day_str} ==========")
        for member in tqdm(range(total_members)):
            # extract data to use
            z_500_1d = v_step[target_var].isel(number=member).compute().values

            # regrid directly on the fly
            grid_lon, grid_lat, z_500_2d = interpolate_to_grid(
                lon_1d, lat_1d, z_500_1d, interp_type="linear", hres=1.25
            )

            lons_axis = grid_lon[0, :]
            lats_axis = grid_lat[:, 0]

            custom_odir = f"./d01_tmp/ts_{current_ts.strftime('%Y%m%d')}_member_{member}"

            # run colindex2
            Detect(
                da=z_500_2d,
                odir=custom_odir,
                ty="L",
                lev=500,
                lons=lons_axis,
                lats=lats_axis,
                t=[current_ts],
                r=np.arange(200, 1001, 100),
                So_thres=3.0,
                SR_thres=3.0,
                Do_thres=0.0,
                xx_thres=0.0,
                rm_rmin=True,
                rm_rmax=True,
                local_ex_req_num=7,
            )

            # read and filter results
            csv_path = f"{custom_odir}/V/{ym_str}/{filename}"
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                if not df.empty:
                    # Filter for bounding box and So >= 6
                    df_peninsula = df[
                        (df["lat"] >= lat_min)
                        & (df["lat"] <= lat_max)
                        & (df["lon"] >= lon_min)
                        & (df["lon"] <= lon_max)
                        & (df["So"] >= 6)
                    ].copy()

                    if not df_peninsula.empty:
                        df_peninsula["EEang_deg"] = np.degrees(df_peninsula["EEang"])
                        df_peninsula["Target_Date"] = current_ts
                        df_peninsula["Forecast_Init"] = forecast_init_date
                        df_peninsula["Member"] = member

                        up_or_down_list = []
                        for _, row in df_peninsula.iterrows():
                            target_lat = lat_1d[np.abs(lat_1d - row["lat"]).argmin()]
                            valid_mask = lat_1d == target_lat

                            if valid_mask.any():
                                max_lon = lon_1d[valid_mask][np.argmax(z_500_1d[valid_mask])]
                                up_or_down_list.append("up" if max_lon > row["lon"] else "down")
                            else:
                                up_or_down_list.append(pd.NA)

                        df_peninsula["up_or_down"] = up_or_down_list

                        clean_df = df_peninsula[[
                            "Target_Date", "Forecast_Init", "Member", "lon", "lat", 
                            "So", "up_or_down", "Do", "ro", "ex", "SR", "m", 
                            "EEang_deg", "EE"
                        ]]
                        final_results.append(clean_df)

            # clean temp folder
            shutil.rmtree(custom_odir, ignore_errors=True)

    # Save original point list
    output_csv = f"cols_index_Init_{forecast_init_date}.csv"
    if final_results:
        master_df = pd.concat(final_results, ignore_index=True)
        master_df.to_csv(output_csv, index=False)
        print(f"Saved raw cut-off detections to {output_csv}")
    else:
        # Create an empty CSV with headers if nothing was detected to prevent errors downstream
        pd.DataFrame(columns=["Target_Date", "Forecast_Init", "Member", "lon", "lat", "So"]).to_csv(output_csv, index=False)
        print("No cut-offs detected. Saved empty CSV.")

    if os.path.exists("_stencil_coords.npy"):
        os.remove("_stencil_coords.npy")
        
    return output_csv, total_members


def compute_col_d4_index(
    detection_csv, forecast_init_date, total_members, 
    lat_min, lat_max, lon_min, lon_max,
    window_start="2026-05-21", window_end="2026-05-25",
    keep_only_up=False
):
    """
    Reads the raw detection CSV, filters by geography and optionally 'up' status,
    and computes the time-averaged 'So' index (COL_d4) for each member using DAILY means.
    """
    target_times = pd.date_range(start=window_start, end=window_end, freq="D")
    total_timesteps = len(target_times)

    # Read the detections
    df = pd.read_csv(detection_csv)
    
    # Apply spatial and conditional filters directly
    if not df.empty:
        df = df[
            (df["lat"] >= lat_min) & 
            (df["lat"] <= lat_max) & 
            (df["lon"] >= lon_min) & 
            (df["lon"] <= lon_max)
        ]
        
        if keep_only_up:
            df = df[df["up_or_down"] == "up"]

    # Group by member and sum the 'So' scores
    if not df.empty:
        so_sums = df.groupby("Member")["So"].sum()
    else:
        so_sums = pd.Series(dtype=float)

    # Build the final index for EVERY member (even those with 0 detections)
    index_data = []
    for mem in range(total_members):
        total_so = so_sums.get(mem, 0.0)  # safely returns 0.0 if member had no cut-offs
        
        index_data.append({
            "Forecast_Init": forecast_init_date,
            "Member": mem,
            "Total_So": total_so,
            "Timesteps": total_timesteps,
            "COL_d4": total_so / total_timesteps
        })

    index_df = pd.DataFrame(index_data)
    
    # Adjust output filename so 'up' specific runs don't overwrite standard runs
    up_tag = "_UP" if keep_only_up else ""
    output_csv = f"COL_d4_index_Init_{forecast_init_date}{up_tag}.csv"
    index_df.to_csv(output_csv, index=False)
    
    print(f"Saved COL_d4 index to {output_csv}")
    return index_df

In [3]:
# setup
base_dir = "/scratchx/pchevali/MAY_2026_HEATWAVE_PROCESSED"
target_var = "z_500"
forecast_init_date = "2026051700"

wide_lat_min, wide_lat_max = 25.0, 50.0
wide_lon_min, wide_lon_max = -35.0, 0.0

csv_path, total_members = detect_cutoffs(
    base_dir=base_dir, 
    forecast_init_date=forecast_init_date, 
    target_var=target_var, 
    lat_min=wide_lat_min, 
    lat_max=wide_lat_max, 
    lon_min=wide_lon_min, 
    lon_max=wide_lon_max
)

# peninsula
pen_lat_min, pen_lat_max = 37.0, 45.0
pen_lon_min, pen_lon_max = -17.0, -2.5

index_df_standard = compute_col_d4_index(
    detection_csv=csv_path, 
    forecast_init_date=forecast_init_date, 
    total_members=total_members,
    lat_min=pen_lat_min, 
    lat_max=pen_lat_max, 
    lon_min=pen_lon_min, 
    lon_max=pen_lon_max
)

index_df_up = compute_col_d4_index(
    detection_csv=csv_path, 
    forecast_init_date=forecast_init_date, 
    total_members=total_members,
    lat_min=pen_lat_min, 
    lat_max=pen_lat_max, 
    lon_min=pen_lon_min, 
    lon_max=pen_lon_max,
    keep_only_up=True
)


========== Processing Target Date: 2026-05-21 ==========


  0%|          | 0/201 [00:00<?, ?it/s]

calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-------

  0%|          | 0/201 [00:00<?, ?it/s]

calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-------

...

calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-------

  0%|          | 0/201 [00:00<?, ?it/s]

calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-------

  0%|          | 0/201 [00:00<?, ?it/s]

calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-------

In [4]:
# setup
base_dir = "/scratchx/pchevali/MAY_2026_HEATWAVE_PROCESSED"
target_var = "z_500"
forecast_init_date = "2026051800"

wide_lat_min, wide_lat_max = 25.0, 50.0
wide_lon_min, wide_lon_max = -35.0, 0.0

csv_path, total_members = detect_cutoffs(
    base_dir=base_dir, 
    forecast_init_date=forecast_init_date, 
    target_var=target_var, 
    lat_min=wide_lat_min, 
    lat_max=wide_lat_max, 
    lon_min=wide_lon_min, 
    lon_max=wide_lon_max
)

# peninsula
pen_lat_min, pen_lat_max = 37.0, 45.0
pen_lon_min, pen_lon_max = -17.0, -2.5

index_df_standard = compute_col_d4_index(
    detection_csv=csv_path, 
    forecast_init_date=forecast_init_date, 
    total_members=total_members,
    lat_min=pen_lat_min, 
    lat_max=pen_lat_max, 
    lon_min=pen_lon_min, 
    lon_max=pen_lon_max
)

index_df_up = compute_col_d4_index(
    detection_csv=csv_path, 
    forecast_init_date=forecast_init_date, 
    total_members=total_members,
    lat_min=pen_lat_min, 
    lat_max=pen_lat_max, 
    lon_min=pen_lon_min, 
    lon_max=pen_lon_max,
    keep_only_up=True
)


========== Processing Target Date: 2026-05-21 ==========


  0%|          | 0/201 [00:00<?, ?it/s]

calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605210000
-------

  0%|          | 0/201 [00:00<?, ?it/s]

calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605220000
-------

...

calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605230000
-------

  0%|          | 0/201 [00:00<?, ?it/s]

calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605240000
-------

  0%|          | 0/201 [00:00<?, ?it/s]

calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-----------------------
done!
calc device: cpu
-----------------------
save 500 202605250000
-------